# 03 — Phase Estimation: Toy Models

**Trilha:** Pensamento espectral  
**Milestone:** 2 — Computação quântica como processamento espectral

---

## Pergunta

Como um circuito extrai fase associada a um autovetor? Por que phase estimation é um algoritmo espectral e não apenas uma sequência de portas?

Este é o primeiro marco real do projeto. QPE não "resolve qualquer problema" — ela extrai estrutura espectral sob hipóteses fortes. Entender exatamente quais hipóteses e o que é extraído é o objetivo deste notebook.

## Modelo matemático mínimo

### Quantum Phase Estimation (QPE)

**Entrada:** unitário $U$ com $U|u\rangle = e^{2\pi i \phi}|u\rangle$, e $n$ qubits de controle.

**Protocolo:**
1. Preparar controles em $H^{\otimes n}|0\rangle^{\otimes n} = \frac{1}{2^{n/2}}\sum_{k=0}^{2^n-1}|k\rangle$
2. Aplicar $\text{ctrl-}U^{2^j}$ para $j = 0, 1, ..., n-1$
3. Aplicar QFT inversa no registrador de controle
4. Medir: resultado é $\tilde{\phi} \approx \phi$ com $n$ bits de precisão

**Após passo 2:** o registrador de controle contém
$$\frac{1}{2^{n/2}} \sum_{k=0}^{2^n-1} e^{2\pi i k \phi} |k\rangle$$

Isso é exatamente a DFT de $\phi$. A QFT inversa desfaz isso e concentra a amplitude em $|\tilde{\phi}\rangle$.

**Hipóteses:**
- O alvo está em autovetor de $U$ (ou próximo disso)
- Conseguimos implementar $U^{2^j}$ de forma eficiente
- $n$ qubits de controle → precisão $2^{-n}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

def qft_matrix(n):
    """Matriz da QFT para n estados."""
    omega = np.exp(2j * np.pi / n)
    return np.array([[omega**(j*k) for k in range(n)] for j in range(n)]) / np.sqrt(n)

def qpe_simulate(phi, n_control_bits):
    """
    Simula QPE para unitário U com autovalor e^(2πiφ).
    Retorna distribuição de probabilidades sobre os 2^n resultados.
    """
    N = 2**n_control_bits
    
    # Após kickback com U^k: amplitude e^(2πi·k·φ)
    state = np.array([np.exp(2j * np.pi * k * phi) for k in range(N)]) / np.sqrt(N)
    
    # QFT inversa
    QFT = qft_matrix(N)
    IQFT = QFT.conj().T
    state_out = IQFT @ state
    
    # Probabilidades
    probs = np.abs(state_out)**2
    return probs

# Caso 1: φ exatamente representável
phi_exact = 3/8  # 3 bits: 0.011 em binário
n_bits = 3
probs = qpe_simulate(phi_exact, n_bits)
outcomes = np.arange(2**n_bits)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(outcomes / 2**n_bits, probs, width=1/2**n_bits * 0.8, color='steelblue')
axes[0].axvline(phi_exact, color='red', linestyle='--', label=f'φ = {phi_exact}')
axes[0].set_xlabel('Valor estimado de φ')
axes[0].set_ylabel('Probabilidade')
axes[0].set_title(f'QPE: φ = {phi_exact} (representável com {n_bits} bits)')
axes[0].legend()

# Caso 2: φ não exatamente representável
phi_inexact = 1/3
probs2 = qpe_simulate(phi_inexact, n_bits)

axes[1].bar(outcomes / 2**n_bits, probs2, width=1/2**n_bits * 0.8, color='coral')
axes[1].axvline(phi_inexact, color='red', linestyle='--', label=f'φ = {phi_inexact:.4f}')
axes[1].set_xlabel('Valor estimado de φ')
axes[1].set_ylabel('Probabilidade')
axes[1].set_title(f'QPE: φ = 1/3 (NÃO representável com {n_bits} bits)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Caso exato: toda amplitude concentrada em {phi_exact}')
print(f'Caso inexato: amplitude distribuída — maior prob em {outcomes[np.argmax(probs2)]/2**n_bits:.4f}')

In [ ]:
# --- Verificação com operador concreto ---
# U = rotação por ângulo θ em 2D
theta = 2 * np.pi * (3/8)  # phase = 3/8
U = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

# Autovalores de U
eigvals = np.linalg.eigvals(U)
phases_exact = np.angle(eigvals) / (2 * np.pi)

print(f'U = rotação por 2π × (3/8) = {np.degrees(theta):.1f}°')
print(f'Autovalores de U: {eigvals.round(4)}')
print(f'Fases (em unidades de 2π): {phases_exact.round(4)}')
print(f'  -> φ₁ = +3/8 = {3/8:.4f}')
print(f'  -> φ₂ = -3/8 ≡ 5/8 = {5/8:.4f}')
print()
print('QPE extrai essas fases — o espectro de U expresso em binário.')

In [ ]:
# --- Precisão cresce com número de bits ---
phi_target = 1/3  # irracional em binário
bit_counts = [2, 3, 4, 5, 6, 8]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, n in enumerate(bit_counts):
    probs = qpe_simulate(phi_target, n)
    N = 2**n
    outcomes = np.arange(N)
    best_estimate = outcomes[np.argmax(probs)] / N
    error = abs(best_estimate - phi_target)
    
    axes[i].bar(outcomes / N, probs, width=0.8/N, color='steelblue', alpha=0.7)
    axes[i].axvline(phi_target, color='red', linestyle='--', alpha=0.8)
    axes[i].set_title(f'n={n} bits | erro={error:.4f}')
    axes[i].set_xlabel('φ estimado')
    axes[i].set_ylabel('Prob')

plt.suptitle('QPE para φ = 1/3: precisão como função do número de qubits de controle', 
             fontsize=12)
plt.tight_layout()
plt.show()

print('Precisão: com n bits, o erro é no máximo 2^(-n).')
print('Cada bit adicional de controle DUPLICA a precisão.')

## Conclusão

1. **QPE é um algoritmo espectral.** Ele não computa uma função arbitrária — extrai a fase associada a um autovetor de um unitário. A saída é informação sobre o espectro de $U$.

2. **O mecanismo é interferência na base de Fourier.** Após os kickbacks, o registrador de controle está no estado $\sum_k e^{2\pi i k\phi}|k\rangle/\sqrt{N}$ — a DFT de $\phi$. QFT inversa transforma isso em amplitude concentrada em $|\tilde{\phi}\rangle$.

3. **Hipóteses fortes:** O alvo precisa ser autovetor (ou próximo disso) de $U$. Precisamos implementar $U^{2^j}$ eficientemente. Sem essas condições, QPE não funciona — ou funciona com custo que elimina a vantagem.

4. **Precisão de $n$ bits custa $n$ qubits de controle e $O(n)$ aplicações de $U$.** Não é mágica — é troca de recurso quântico (qubits + portas) por precisão.

5. **O que QPE NÃO faz:** não resolve problemas arbitrários, não acha autovalores de matrizes arbitrárias sem preparação de autovetores, não extrai espectro completo em uma rodada.

---

**Próximo:** Hamiltonian simulation — como implementar $e^{-iHt}$ eficientemente para que QPE possa extrair os autovalores de $H$.